# Hospital Management Data Analytics & Predictive Dashboard
**Author:** Jagruti Sahu | **Program:** IBM SkillsBuild Data Analytics with AI Internship 2026 (BharatCares / AICTE)

**Datasets:** `patients.csv`, `doctors.csv`, `appointments.csv`, `treatments.csv`, `billing.csv`

**Pipeline:** load → clean → merge → KPIs → charts → 3 ML models → revenue forecast → conclusions.
The interactive frontend is in `dashboard.py` (Dash + Plotly).

## 1. Load data

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path(".")          # folder containing the 5 CSV files
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 100})

patients = pd.read_csv(DATA/"patients.csv", parse_dates=["date_of_birth", "registration_date"],
                       keep_default_na=False, na_values=[""])
doctors  = pd.read_csv(DATA/"doctors.csv")
appts    = pd.read_csv(DATA/"appointments.csv", parse_dates=["appointment_date"])
treat    = pd.read_csv(DATA/"treatments.csv", parse_dates=["treatment_date"])
bill     = pd.read_csv(DATA/"billing.csv", parse_dates=["bill_date"])
for name, df in dict(patients=patients, doctors=doctors, appointments=appts, treatments=treat, billing=bill).items():
    print(f"{name:<13}{df.shape}")

## 2. Data quality checks and cleaning

In [ ]:
keys = {"patients": (patients, "patient_id"), "doctors": (doctors, "doctor_id"), "appointments": (appts, "appointment_id"),
        "treatments": (treat, "treatment_id"), "billing": (bill, "bill_id")}
print("=== Quality report (before cleaning) ===")
for n, (df, k) in keys.items():
    print(f"{n:<13} rows={len(df):>4}  nulls={int(df.isna().sum().sum()):>3}  duplicate_keys={int(df[k].duplicated().sum())}")
print("orphan appointments (patient):", (~appts.patient_id.isin(patients.patient_id)).sum())
print("orphan appointments (doctor): ", (~appts.doctor_id.isin(doctors.doctor_id)).sum())
print("orphan treatments:            ", (~treat.appointment_id.isin(appts.appointment_id)).sum())
print("orphan bills:                 ", (~bill.treatment_id.isin(treat.treatment_id)).sum())

# cleaning
for df in (patients, doctors, appts, treat, bill):
    for c in df.select_dtypes("object"):
        df[c] = df[c].astype(str).str.strip()
patients = patients.drop_duplicates("patient_id"); doctors = doctors.drop_duplicates("doctor_id")
appts = appts.drop_duplicates("appointment_id");   treat = treat.drop_duplicates("treatment_id")
bill = bill.drop_duplicates("bill_id")
bill["amount"] = pd.to_numeric(bill["amount"], errors="coerce"); bill = bill[bill.amount > 0]
treat["cost"] = pd.to_numeric(treat["cost"], errors="coerce")
patients["gender"] = patients["gender"].str.upper()
ref = pd.Timestamp.today()
patients["age"] = ((ref - patients.date_of_birth).dt.days // 365).astype(int)
patients["age_group"] = pd.cut(patients.age, [0, 17, 35, 50, 65, 120], labels=["0-17", "18-35", "36-50", "51-65", "65+"])
appts["weekday"] = appts.appointment_date.dt.day_name()
print("\nCleaning done.")

## 3. Merge into one master table

In [ ]:
doctors["doctor_name"] = "Dr. " + doctors.first_name + " " + doctors.last_name
master = (appts
    .merge(patients[["patient_id", "gender", "age", "age_group", "insurance_provider"]], on="patient_id", how="left")
    .merge(doctors[["doctor_id", "doctor_name", "specialization", "hospital_branch"]], on="doctor_id", how="left")
    .merge(treat[["treatment_id", "appointment_id", "treatment_type", "cost"]], on="appointment_id", how="left")
    .merge(bill[["bill_id", "treatment_id", "amount", "payment_method", "payment_status", "bill_date"]], on="treatment_id", how="left"))
print(master.shape); master.head()

## 4. KPIs

In [ ]:
paid = bill.loc[bill.payment_status == "Paid", "amount"].sum()
kpi = {
    "Total patients": patients.patient_id.nunique(),
    "Total doctors": doctors.doctor_id.nunique(),
    "Total appointments": len(appts),
    "Completion rate %": round((appts.status == "Completed").mean() * 100, 1),
    "Cancellation rate %": round((appts.status == "Cancelled").mean() * 100, 1),
    "No-show rate %": round((appts.status == "No-show").mean() * 100, 1),
    "Total billed": round(bill.amount.sum(), 2),
    "Revenue collected (Paid)": round(paid, 2),
    "Pending amount": round(bill.loc[bill.payment_status == "Pending", "amount"].sum(), 2),
    "Failed amount": round(bill.loc[bill.payment_status == "Failed", "amount"].sum(), 2),
    "Collection rate %": round(paid / bill.amount.sum() * 100, 1),
    "Average bill": round(bill.amount.mean(), 2),
}
pd.DataFrame(kpi, index=["value"]).T

## 5. Exploratory charts

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(17, 9))
appts.status.value_counts().plot.bar(ax=ax[0, 0], color="#2a6fdb", title="Appointments by status"); ax[0, 0].tick_params(axis="x", rotation=0)
appts.groupby(appts.appointment_date.dt.to_period("M")).size().plot(ax=ax[0, 1], color="#2a6fdb", title="Monthly appointments")
bill.merge(treat[["treatment_id", "treatment_type"]], on="treatment_id").groupby("treatment_type").amount.sum().sort_values().plot.barh(ax=ax[0, 2], color="#1b9e77", title="Revenue by treatment type")
pd.crosstab(bill.payment_method, bill.payment_status).plot.bar(stacked=True, ax=ax[1, 0], title="Payment method vs status"); ax[1, 0].tick_params(axis="x", rotation=0)
master.groupby("specialization").amount.sum().sort_values().plot.barh(ax=ax[1, 1], color="#d95f02", title="Revenue by specialization")
patients.age_group.value_counts().sort_index().plot.bar(ax=ax[1, 2], color="#66a61e", title="Patients by age group"); ax[1, 2].tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
top = (master.groupby("doctor_name").agg(appointments=("appointment_id", "count"),
        completed=("status", lambda s: (s == "Completed").sum()), revenue=("amount", "sum"))
       .assign(completion_pct=lambda d: (d.completed / d.appointments * 100).round(1)).sort_values("revenue", ascending=False))
top.head(10).round(0)

## 6. Model 1 – Bill amount prediction (regression)

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, r2_score, roc_auc_score, accuracy_score, f1_score, classification_report

d = master.dropna(subset=["amount"]).copy()
num, cat = ["age"], ["gender", "insurance_provider", "specialization", "treatment_type", "reason_for_visit"]
X, y = d[num + cat], d["amount"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
enc = ColumnTransformer([("c", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat)], remainder="passthrough")
gbr = Pipeline([("enc", enc), ("m", GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42))]).fit(Xtr, ytr)
base = DummyRegressor(strategy="mean").fit(Xtr, ytr)
print(f"Gradient Boosting  MAE={mean_absolute_error(yte, gbr.predict(Xte)):,.0f}  R2={r2_score(yte, gbr.predict(Xte)):.3f}")
print(f"Mean baseline      MAE={mean_absolute_error(yte, base.predict(Xte)):,.0f}  R2={r2_score(yte, base.predict(Xte)):.3f}")
print("Rows used:", len(d), "| a negative R2 means the model is worse than predicting the average bill.")

## 7. Model 2 – Appointment no-show prediction (classification)

In [ ]:
d = master[master.status.isin(["Completed", "No-show"])].copy()
d["y"] = (d.status == "No-show").astype(int)
cat2 = ["gender", "insurance_provider", "specialization", "reason_for_visit", "weekday"]
X, y = d[["age"] + cat2], d["y"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
enc2 = ColumnTransformer([("c", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat2)], remainder="passthrough")
rf = Pipeline([("enc", enc2), ("m", RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=3, class_weight="balanced", random_state=42))]).fit(Xtr, ytr)
print("Rows:", len(d), "| No-show:", int(y.sum()), "| Completed:", int((1 - y).sum()))
print("Test ROC-AUC:", round(roc_auc_score(yte, rf.predict_proba(Xte)[:, 1]), 3))
cv = cross_val_score(rf, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="roc_auc")
print(f"5-fold CV ROC-AUC: {cv.mean():.3f} +/- {cv.std():.3f}   (0.5 = random guessing)")
print(classification_report(yte, rf.predict(Xte), target_names=["Show", "No-show"]))

## 8. Model 3 – Payment status prediction (Paid / Pending / Failed)

In [ ]:
d = master.dropna(subset=["amount"]).copy()
cat3 = ["payment_method", "treatment_type", "insurance_provider"]
X, y = d[["amount"] + cat3], d["payment_status"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
def make(model, scale=False):
    pre = ColumnTransformer([("c", OneHotEncoder(handle_unknown="ignore"), cat3),
                             ("n", StandardScaler() if scale else "passthrough", ["amount"])])
    return Pipeline([("pre", pre), ("m", model)])
models = {"Logistic Regression": make(LogisticRegression(max_iter=1000, class_weight="balanced"), True),
          "Random Forest": make(RandomForestClassifier(n_estimators=300, max_depth=6, class_weight="balanced", random_state=42)),
          "Gradient Boosting": make(GradientBoostingClassifier(random_state=42))}
rows = []
for n, m in models.items():
    m.fit(Xtr, ytr); p = m.predict(Xte)
    cvf = cross_val_score(m, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="f1_weighted").mean()
    rows.append([n, accuracy_score(yte, p), f1_score(yte, p, average="weighted"), cvf])
res = pd.DataFrame(rows, columns=["Model", "Accuracy", "Test F1 (weighted)", "CV F1 (weighted)"]).round(3)
print("Class balance:", y.value_counts().to_dict(), "| majority-class baseline accuracy:", round(y.value_counts(normalize=True).max(), 3))
res

## 9. Revenue forecasting (next 6 months)

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
m = bill.groupby(bill.bill_date.dt.to_period("M")).amount.sum().sort_index()
m.index = m.index.to_timestamp(); m = m.asfreq("MS").fillna(0)
H, TEST = 6, 3
if len(m) < 8:
    print("Not enough monthly history for forecasting (need at least 8 months).")
else:
    t = np.arange(len(m))
    tr, te = m.iloc[:-TEST], m.iloc[-TEST:]
    ttr, tte = t[:-TEST], t[-TEST:]
    def fc_lr(y, tt, tf):   return LinearRegression().fit(tt.reshape(-1, 1), y).predict(tf.reshape(-1, 1))
    def fc_hw(y, tt, tf):   return np.asarray(ExponentialSmoothing(y, trend="add").fit().forecast(len(tf)))
    def fc_rf(y, tt, tf):
        from sklearn.ensemble import RandomForestRegressor
        Xa = pd.DataFrame({"t": tt}); Xb = pd.DataFrame({"t": tf})
        return RandomForestRegressor(200, min_samples_leaf=2, random_state=42).fit(Xa, y).predict(Xb)
    def fc_mean(y, tt, tf): return np.repeat(y.mean(), len(tf))
    methods = {"Mean baseline": fc_mean, "Linear trend": fc_lr, "Holt-Winters": fc_hw, "Random Forest": fc_rf}
    out = []
    for n, f in methods.items():
        p = f(tr.values, ttr, tte)
        out.append([n, mean_absolute_error(te, p), (abs(te.values - p) / te.values).mean() * 100])
    bt = pd.DataFrame(out, columns=["Method", "MAE", "MAPE %"]).round(1).sort_values("MAE")
    print(f"History: {len(m)} months | backtest on last {TEST} months"); display(bt)
    best = bt.iloc[0]["Method"]; print("Best by MAE:", best)
    tf = np.arange(len(m), len(m) + H); fidx = pd.date_range(m.index[-1] + pd.offsets.MonthBegin(), periods=H, freq="MS")
    fut = pd.Series(methods[best](m.values, t, tf), index=fidx)
    ax = m.plot(figsize=(10, 4), marker="o", label="Historical billed"); fut.plot(ax=ax, marker="s", ls="--", color="tab:red", label=f"Forecast ({best})")
    ax.fill_between(fut.index, fut * 0.85, fut * 1.15, color="tab:red", alpha=0.12, label="+/-15% band (indicative)")
    ax.set_title("Monthly billed revenue - history and 6-month forecast"); ax.legend(); plt.show()
    display(fut.round(0).to_frame("Forecast").T)

## 10. Conclusions and recommendations
* **Operations:** cancellations and no-shows lose a large share of booked slots. Send SMS/WhatsApp reminders 24 hours before appointments and consider limited overbooking for high-risk slots.
* **Revenue cycle:** collection rate is well below 100 percent. Follow up on Pending and Failed bills and review the payment methods with the most failures.
* **Doctor workload:** rebalance appointments away from the busiest doctors and track completion rate next to revenue.
* **Models (honest results):** compare each model with its baseline above. On small datasets (a few hundred rows), the no-show, payment-status and bill-amount models often perform close to chance or below the mean baseline. This means the pipeline is sound, but richer features (booking lead time, payment history, itemised procedure codes) and more data are needed before any model is used for real decisions.
* **Forecast:** with only about a year of monthly data, treat forecasts as rough planning ranges and re-check monthly against actual billing.
* **Privacy:** real patient data must be masked/anonymised and handled under HIPAA / local regulations.